In [3]:
print("Super El Niño Sri Lanka Project Started")

Super El Niño Sri Lanka Project Started


In [4]:
#Import Libraries
import pandas as pd
import requests
from pathlib import Path

In [5]:
#Download MEI Dataset
url = "https://psl.noaa.gov/enso/mei/data/meiv2.data"

response = requests.get(url)

print("Status Code:", response.status_code)

print(response.text[:500])

Status Code: 200
1979     2026
1979     0.46     0.29    -0.05     0.22     0.27    -0.11    -0.10     0.47     0.38     0.23     0.53     0.63
1980     0.33     0.21     0.39     0.50     0.45     0.55     0.47     0.03     0.12     0.03    -0.08    -0.12
1981    -0.35    -0.23     0.33     0.42    -0.24    -0.69    -0.60    -0.33    -0.06    -0.16    -0.27    -0.20
1982    -0.43    -0.50    -0.28    -0.37    -0.11     0.63     1.65     1.91     1.68     1.77     2.13     2.36
1983     2.47     2.66     2.60   


In [6]:
#Save Raw File
raw_path = Path("../data/raw/meiv2.data")

with open(raw_path, "w") as f:
    f.write(response.text)

print("File saved successfully!")

File saved successfully!


In [7]:
import pandas as pd

file_path = "../data/raw/meiv2.data"

# Read raw lines
with open(file_path, "r") as f:
    lines = f.readlines()

# Extract yearly data rows
data_rows = []

for line in lines:
    parts = line.strip().split()

    if len(parts) == 13:
        if parts[0].isdigit():
            data_rows.append(parts)

print("Years Found:", len(data_rows))
print(data_rows[:3])

Years Found: 48
[['1979', '0.46', '0.29', '-0.05', '0.22', '0.27', '-0.11', '-0.10', '0.47', '0.38', '0.23', '0.53', '0.63'], ['1980', '0.33', '0.21', '0.39', '0.50', '0.45', '0.55', '0.47', '0.03', '0.12', '0.03', '-0.08', '-0.12'], ['1981', '-0.35', '-0.23', '0.33', '0.42', '-0.24', '-0.69', '-0.60', '-0.33', '-0.06', '-0.16', '-0.27', '-0.20']]


In [8]:
months = [
    "01","02","03","04","05","06",
    "07","08","09","10","11","12"
]

records = []

for row in data_rows:

    year = int(row[0])

    for i in range(12):

        mei = float(row[i+1])

        # Skip missing values
        if mei == -999.00:
            continue

        records.append([
            f"{year}-{months[i]}",
            mei
        ])

mei_df = pd.DataFrame(
    records,
    columns=["Date", "MEI"]
)

print(mei_df.shape)

mei_df.head()

(569, 2)


,Date,MEI
0,1979-01,0.46
1,1979-02,0.29
2,1979-03,-0.05
3,1979-04,0.22
4,1979-05,0.27


In [9]:
mei_df.tail()

,Date,MEI
564,2026-01,-0.76
565,2026-02,-0.95
566,2026-03,-1.03
567,2026-04,-0.64
568,2026-05,0.27


In [10]:
output_path = "../data/processed/mei.csv"

mei_df.to_csv(output_path, index=False)

print("MEI dataset saved successfully!")
print(f"Rows: {len(mei_df)}")

MEI dataset saved successfully!
Rows: 569


In [11]:
mei_df.describe()

,MEI
count,569.000000
mean,-0.081652
std,0.970575
min,-2.420000
25%,-0.770000
50%,-0.160000
75%,0.470000
max,2.850000


In [12]:
import requests

LAT = 6.9271
LON = 79.8612

url = (
    "https://power.larc.nasa.gov/api/temporal/monthly/point"
    "?parameters=PRECTOTCORR"
    "&community=AG"
    "&longitude=79.8612"
    "&latitude=6.9271"
    "&start=2000"
    "&end=2025"
    "&format=JSON"
)

response = requests.get(url)

print("Status:", response.status_code)

print(response.text[:1000])

Status: 200
{"type":"Feature","geometry":{"type":"Point","coordinates":[79.861,6.927,123.66]},"properties":{"parameter":{"PRECTOTCORR":{"200001":0.94,"200002":2.94,"200003":2.07,"200004":3.65,"200005":2.15,"200006":3.52,"200007":2.61,"200008":5.37,"200009":3.21,"200010":3.08,"200011":6.73,"200012":2.81,"200013":3.25,"200101":3.73,"200102":2.25,"200103":0.46,"200104":6.9,"200105":4.32,"200106":4.21,"200107":3.01,"200108":0.69,"200109":4.96,"200110":3.36,"200111":4.27,"200112":2.77,"200113":3.4,"200201":1.07,"200202":2.21,"200203":1.78,"200204":6.88,"200205":6.79,"200206":2.52,"200207":1.07,"200208":2.32,"200209":1.67,"200210":12.35,"200211":7.02,"200212":5.46,"200213":4.28,"200301":1.9,"200302":2.51,"200303":3.9,"200304":4.65,"200305":5.73,"200306":3.85,"200307":2.85,"200308":1.81,"200309":3.01,"200310":4.36,"200311":9.99,"200312":0.82,"200313":3.77,"200401":0.81,"200402":0.91,"200403":1.01,"200404":4.34,"200405":6.4,"200406":3.61,"200407":2.38,"200408":1.86,"200409":7.92,"200410":8.23,

In [13]:
response.json().keys()

dict_keys(['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times'])

In [14]:
rainfall_data = response.json()["properties"]["parameter"]["PRECTOTCORR"]

print(type(rainfall_data))

list(rainfall_data.items())[:5]

<class 'dict'>


[('200001', 0.94),
 ('200002', 2.94),
 ('200003', 2.07),
 ('200004', 3.65),
 ('200005', 2.15)]

In [15]:
rain_df = pd.DataFrame(
    rainfall_data.items(),
    columns=["Date", "Rainfall"]
)

rain_df.head()

,Date,Rainfall
0,200001,0.94
1,200002,2.94
2,200003,2.07
3,200004,3.65
4,200005,2.15


In [16]:
rain_df.info()# Keep only valid months (01-12)

rain_df = rain_df[
    ~rain_df["Date"].str.endswith("13")
].copy()

print("Rows after cleaning:", len(rain_df))

rain_df.tail()

<class 'pandas.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      338 non-null    str    
 1   Rainfall  338 non-null    float64
dtypes: float64(1), str(1)
memory usage: 7.4 KB
Rows after cleaning: 312


,Date,Rainfall
332,202508,2.64
333,202509,5.96
334,202510,16.80
335,202511,20.90
336,202512,4.62


In [17]:
rain_df["Date"] = pd.to_datetime(
    rain_df["Date"],
    format="%Y%m"
)

rain_df.info()

<class 'pandas.DataFrame'>
Index: 312 entries, 0 to 336
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      312 non-null    datetime64[us]
 1   Rainfall  312 non-null    float64       
dtypes: datetime64[us](1), float64(1)
memory usage: 7.3 KB


In [18]:
print(rain_df.shape)

rain_df.head()


(312, 2)


,Date,Rainfall
0,2000-01-01,0.94
1,2000-02-01,2.94
2,2000-03-01,2.07
3,2000-04-01,3.65
4,2000-05-01,2.15


In [19]:
rain_df.tail()

,Date,Rainfall
332,2025-08-01,2.64
333,2025-09-01,5.96
334,2025-10-01,16.80
335,2025-11-01,20.90
336,2025-12-01,4.62


In [20]:
rain_df.to_csv(
    "../data/processed/colombo_rainfall.csv",
    index=False
)

print("Colombo rainfall dataset saved!")

Colombo rainfall dataset saved!


In [21]:
print("Done")

Done
